# h-Adaptive Refinement on the L-Shaped Domain

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/camlab-ethz/TensorMesh/blob/main/notebooks/poisson_h_adaptivity.ipynb)

The re-entrant corner of an L-shaped domain makes the exact solution
$u = r^{2/3}\sin(2\theta/3)$ singular in its gradient — the canonical test
case for adaptive mesh refinement. Uniform refinement wastes elements far
from the corner and converges sub-optimally; **solve → estimate → mark →
remesh** recovers the optimal rate.

The error indicator is a flux-jump estimator built from TensorMesh's shape
gradients and facet adjacency, and each level re-meshes through gmsh with a
spatially varying size field.

⏱️ *A few minutes on Colab's free CPU runtime — most of it in the gmsh remeshes.*

Docs: [h-Adaptive Refinement](https://docs.tensor-mesh.com/example_gallery/poisson.html#h-adaptive-refinement-on-the-l-shape-poisson-h-adaptivity-py) · Source: [`examples/poisson/poisson_h_adaptivity.py`](https://github.com/camlab-ethz/TensorMesh/blob/main/examples/poisson/poisson_h_adaptivity.py)

In [ ]:
# Install TensorMesh (skipped automatically if it is already available, e.g. a local dev setup).
# The apt line provides the OpenGL utility library that gmsh -- TensorMesh's mesh generator --
# needs at import time; it is a no-op where the library is already present.
import importlib.util
if importlib.util.find_spec("tensormesh") is None:
    !apt-get -qq install -y libglu1-mesa > /dev/null 2>&1 || true
    %pip install -q tensormesh-fem==0.2.0

## Problem and solver

$-\Delta u = 0$ on the L-shape, with Dirichlet data taken from the exact
singular solution so the error can be measured exactly.

In [ ]:
import contextlib
import os
import tempfile

import gmsh
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.collections import PolyCollection
from scipy.spatial import cKDTree

from tensormesh import (Condenser, LaplaceElementAssembler, MassElementAssembler,
                        Mesh, Transformation)


@contextlib.contextmanager
def quiet():
    """Silence gmsh's progress log — it writes straight to file descriptor 1,
    so this loop would otherwise bury the results under a wall of ``Info :``."""
    with open(os.devnull, "w") as devnull:
        saved = os.dup(1)
        os.dup2(devnull.fileno(), 1)
        try:
            yield
        finally:
            os.dup2(saved, 1)
            os.close(saved)


CORNER = (0.5, 0.5)   # re-entrant corner of Mesh.gen_L()
ALPHA = 2.0 / 3.0     # singularity exponent (pi / omega, omega = 3 pi / 2)


def singular_solution(points, corner=CORNER):
    """u = r^{2/3} sin(2 theta / 3), centred on the re-entrant corner."""
    dx = points[:, 0] - corner[0]
    dy = points[:, 1] - corner[1]
    r = torch.sqrt(dx ** 2 + dy ** 2)
    phi = (torch.atan2(dy, dx) - torch.pi / 2) % (2 * torch.pi)
    return r.pow(ALPHA) * torch.sin(ALPHA * phi)


def solve_laplace(mesh):
    """Solve -Laplace(u) = 0 with u = g on the boundary."""
    K = LaplaceElementAssembler.from_mesh(mesh)(mesh.points)
    g = singular_solution(mesh.points)
    condenser = Condenser(mesh.boundary_mask, g)          # non-zero Dirichlet
    K_, f_ = condenser(K, torch.zeros(mesh.n_points))
    return condenser.recover(K_.solve(f_))


def global_l2_error(mesh, u_fem, u_exact):
    """Mass-weighted relative L2 error."""
    M = MassElementAssembler.from_mesh(mesh)(mesh.points)
    e = u_fem - u_exact
    err2 = torch.abs((e * (M @ e)).sum())
    ref2 = torch.abs((u_exact * (M @ u_exact)).sum())
    return (err2.sqrt() / (ref2.sqrt() + 1e-30)).item()

## Error estimation and marking

Dörfler marking: refine every element whose indicator exceeds
$\theta\,\max_K \eta_K$, and let the very best elements coarsen slightly.

In [ ]:
def element_error_and_sizes(mesh, u_fem):
    """Flux-jump indicator and element diameter for P1 triangles.

    For linear triangles the cell residual of the Laplace problem vanishes, so
    the estimator is driven entirely by jumps of the normal gradient across
    interior edges:  eta_K^2 = sum_e |e|^2 [[grad(u_h) . n]]^2.
    """
    cells = mesh.cells["triangle"]
    points = mesh.points
    coords = points[cells]
    values = u_fem[cells]

    centroids = coords.mean(dim=1)
    diffs = coords.unsqueeze(2) - coords.unsqueeze(1)
    h = diffs.norm(dim=-1).amax(dim=(1, 2))

    # One constant gradient per element, from TensorMesh shape gradients.
    trans = Transformation(points, cells, "triangle", quadrature_order=1)
    grads = torch.einsum("eb,eqbd->eqd", values, trans.shape_grad).mean(dim=1)

    # Interior neighbour pairs via facet adjacency.
    adjacency = mesh.element_adjacency("triangle")
    left, right = adjacency.row, adjacency.col
    unique_pair = left < right
    left, right = left[unique_pair], right[unique_pair]

    left_cells, right_cells = cells[left], cells[right]
    shared_mask = left_cells[:, :, None] == right_cells[:, None, :]
    shared = left_cells[shared_mask.any(dim=2)].reshape(-1, 2)

    edge_vec = points[shared[:, 1]] - points[shared[:, 0]]
    edge_length = edge_vec.norm(dim=1)
    normal = torch.stack([-edge_vec[:, 1], edge_vec[:, 0]], dim=1) / edge_length[:, None]
    jump = ((grads[left] - grads[right]) * normal).sum(dim=1)
    contribution = edge_length.pow(2) * jump.pow(2)

    eta2 = torch.zeros(cells.shape[0], dtype=points.dtype)
    eta2.scatter_add_(0, left, contribution)
    eta2.scatter_add_(0, right, contribution)

    return (centroids.numpy(), eta2.clamp_min(0).sqrt().numpy(), h.numpy())


def doerfler_sizes(h, eta, theta=0.5, refine=0.5, coarsen=1.3, h_min=0.002, h_max=0.15):
    """Mark elements whose error exceeds theta * max(eta) and halve their size."""
    eta_max = eta.max()
    h_new = h.copy()
    h_new[eta > theta * eta_max] *= refine
    h_new[eta < 0.05 * eta_max] *= coarsen
    return np.clip(h_new, h_min, h_max)

## Remeshing

The marked sizes become a gmsh size callback, so each level produces a
genuinely new mesh rather than a bisection of the old one.

In [ ]:
def remesh_L(centroids, sizes, h_min=0.002, h_max=0.15):
    """Re-generate the L-shaped mesh with a spatially varying size field."""
    tree = cKDTree(centroids)

    gmsh.initialize()
    gmsh.option.setNumber("General.Terminal", 0)
    gmsh.model.add("L_adaptive")

    # geometry: [0,1]^2 minus [0.5,1] x [0.5,1]
    r_out = gmsh.model.occ.addRectangle(0, 0, 0, 1, 1)
    r_cut = gmsh.model.occ.addRectangle(0.5, 0.5, 0, 0.5, 0.5)
    gmsh.model.occ.synchronize()
    gmsh.model.occ.cut([(2, r_out)], [(2, r_cut)])
    gmsh.model.occ.synchronize()

    s = gmsh.model.getEntities(2)[0][1]
    bnd = gmsh.model.getBoundary([(2, s)], oriented=False)
    gmsh.model.addPhysicalGroup(1, [l[1] for l in bnd], name="boundary")
    gmsh.model.addPhysicalGroup(2, [s], name="domain")

    def size_cb(dim, tag, x, y, z, lc):
        _, idx = tree.query([x, y])
        return max(h_min, min(h_max, float(sizes[idx])))

    gmsh.model.mesh.setSizeCallback(size_cb)
    gmsh.model.mesh.generate(2)

    tmp = tempfile.NamedTemporaryFile(suffix=".msh", delete=False)
    tmp.close()
    gmsh.write(tmp.name)
    gmsh.finalize()

    mesh = Mesh.from_file(tmp.name, reorder=True)
    os.unlink(tmp.name)

    pts = mesh.points
    eps = 1e-10
    is_boundary = (
        (pts[:, 0] < eps) | (pts[:, 0] > 1 - eps)
        | (pts[:, 1] < eps) | (pts[:, 1] > 1 - eps)
        | ((pts[:, 0] - 0.5).abs() < eps) & (pts[:, 1] > 0.5 - eps)
        | ((pts[:, 1] - 0.5).abs() < eps) & (pts[:, 0] > 0.5 - eps)
    )
    mesh.register_point_data("is_boundary", is_boundary)
    return mesh

## The adaptive loop

In [ ]:
H0 = 0.08
MAX_LEVELS = 6        # the full example runs up to 10
TARGET_ERROR = 5e-4

with quiet():
    mesh = Mesh.gen_L(chara_length=H0, element_type="tri")
adapt_dofs, adapt_errs = [], []
final_mesh, final_u = mesh, None

print(f"{'Level':>5} {'DOFs':>8} {'Elems':>8} {'Rel L2':>14}")
print("-" * 40)
for level in range(MAX_LEVELS):
    u_fem = solve_laplace(mesh)
    u_exact = singular_solution(mesh.points)
    rel_err = global_l2_error(mesh, u_fem, u_exact)

    adapt_dofs.append(mesh.n_points)
    adapt_errs.append(rel_err)
    final_mesh, final_u = mesh, u_fem
    print(f"{level:>5} {mesh.n_points:>8} {mesh.n_elements:>8} {rel_err:>14.4e}")
    if rel_err < TARGET_ERROR:
        print(f"\nConverged at level {level}.")
        break

    centroids, eta, h = element_error_and_sizes(mesh, u_fem)
    h_new = doerfler_sizes(h, eta, theta=0.5, h_min=0.002, h_max=H0)
    with quiet():
        mesh = remesh_L(centroids, h_new, h_min=0.002, h_max=H0)

In [ ]:
print("Uniform refinement, for comparison:")
uni_dofs, uni_errs = [], []
for uh in [0.08, 0.04, 0.02, 0.013]:
    with quiet():
        m = Mesh.gen_L(chara_length=uh, element_type="tri")
    u = solve_laplace(m)
    err = global_l2_error(m, u, singular_solution(m.points))
    uni_dofs.append(m.n_points)
    uni_errs.append(err)
    print(f"  h={uh:.3f}  DOFs={m.n_points:>7}  Rel L2={err:.4e}")

## Results

Plotted per degree of freedom, the adaptive curve should sit clearly below
the uniform one, and the final mesh should show a dense cluster of elements
spiralling into the re-entrant corner.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

# (a) convergence: adaptive vs uniform, per DOF
ax = axes[0]
ax.loglog(adapt_dofs, adapt_errs, "o-", color="#E74C3C", lw=2, ms=7, label="Adaptive")
ax.loglog(uni_dofs, uni_errs, "s--", color="#3498DB", lw=2, ms=7, label="Uniform")
d = np.array(uni_dofs, dtype=float)
ax.loglog(d, uni_errs[0] * d[0] / d, ":", color="gray", lw=1, label=r"$\mathcal{O}(N^{-1})$")
ax.set_xlabel("Number of DOFs")
ax.set_ylabel(r"Relative $L^2$ error")
ax.set_title("Convergence: L-domain singularity")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, which="both")

# (b) the final adaptive mesh -- note the clustering at the corner
ax = axes[1]
for _, cells in final_mesh.cells.items():
    verts = final_mesh.points.numpy()[cells.numpy()]
    ax.add_collection(PolyCollection(verts, edgecolors="black", facecolors="white", lw=0.2))
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02); ax.set_aspect("equal")
ax.set_title(f"Adaptive mesh ({final_mesh.n_points} DOFs)")

# (c) the solution
ax = axes[2]
for _, cells in final_mesh.cells.items():
    c_np = cells.numpy()
    verts = final_mesh.points.numpy()[c_np]
    vals = final_u.detach().numpy()[c_np].mean(axis=1)
    poly = PolyCollection(verts, array=vals, cmap="RdBu_r", edgecolors="none")
    ax.add_collection(poly)
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02); ax.set_aspect("equal")
ax.set_title("FEM solution")
fig.colorbar(poly, ax=ax, shrink=0.8)

fig.tight_layout()
plt.show()